# Tutorial 07: End-to-End Integration Test

This notebook tests **ALL AWP capabilities** end-to-end with real LLM calls.
It serves as both a comprehensive integration test and a demonstration of every
major feature in the Agent Workflow Protocol stack.

**What is tested:**
1. LLM connectivity (text + structured JSON output)
2. Tool calling with the AWP ToolRegistry
3. DAG workflows (Hello World + Research Pipeline)
4. Delegation Loop (simple task + code execution mode)
5. AgentWorkflow high-level API (DataFrame analysis + tool creation)

**Prerequisites:**
- `pip install -e reference/python/`
- An LLM API key (OpenRouter recommended, or any OpenAI-compatible endpoint)

## 1. Provider Setup

Configure your LLM provider. Supported options:
- **Ollama** -- local, no API key needed (install from ollama.com)
- **OpenRouter** -- cloud, requires `OPENROUTER_API_KEY`
- **Custom API** -- any OpenAI-compatible endpoint

In [ ]:
# ============================================================
# Provider Selection -- choose ONE of: "ollama", "openrouter", "custom"
# ============================================================
PROVIDER = "openrouter"  # <-- change this

# --- Ollama (local) -------------------------------------------
OLLAMA_MODEL = "qwen3:1.7b"
OLLAMA_BASE_URL = "http://localhost:11434/v1"

# --- OpenRouter (cloud) ---------------------------------------
OPENROUTER_API_KEY = ""                 # paste your key or set env var OPENROUTER_API_KEY
OPENROUTER_MODEL = "openai/gpt-5-nano"

# --- Custom OpenAI-compatible API -----------------------------
CUSTOM_API_KEY = ""
CUSTOM_BASE_URL = ""
CUSTOM_MODEL = ""

# ==============================================================
# DO NOT EDIT BELOW -- wires up the selected provider
# ==============================================================
import os

if PROVIDER == "ollama":
    os.environ["LLM_API_KEY"] = "ollama"
    os.environ["LLM_BASE_URL"] = OLLAMA_BASE_URL
    MODEL = f"ollama/{OLLAMA_MODEL}"
    print(f"Using Ollama  model={OLLAMA_MODEL}  url={OLLAMA_BASE_URL}")

elif PROVIDER == "openrouter":
    key = OPENROUTER_API_KEY or os.getenv("OPENROUTER_API_KEY", "")
    if not key:
        raise ValueError("Set OPENROUTER_API_KEY above or as an environment variable")
    os.environ["LLM_API_KEY"] = key
    os.environ["LLM_BASE_URL"] = "https://openrouter.ai/api/v1"
    MODEL = OPENROUTER_MODEL
    print(f"Using OpenRouter  model={OPENROUTER_MODEL}")

elif PROVIDER == "custom":
    key = CUSTOM_API_KEY or os.getenv("LLM_API_KEY", "")
    url = CUSTOM_BASE_URL or os.getenv("LLM_BASE_URL", "")
    if not key:
        raise ValueError("Set CUSTOM_API_KEY above or LLM_API_KEY as an environment variable")
    if not url:
        raise ValueError("Set CUSTOM_BASE_URL above or LLM_BASE_URL as an environment variable")
    os.environ["LLM_API_KEY"] = key
    os.environ["LLM_BASE_URL"] = url
    MODEL = CUSTOM_MODEL
    print(f"Using custom API  model={CUSTOM_MODEL}  url={url}")

else:
    raise ValueError(f"Unknown PROVIDER '{PROVIDER}'. Use 'ollama', 'openrouter', or 'custom'.")

# Set the model env var so WorkflowRunner picks it up
os.environ["LLM_MODEL"] = MODEL
print(f"LLM_MODEL={MODEL}")
print(f"LLM_BASE_URL={os.environ['LLM_BASE_URL']}")

In [ ]:
# Common imports, logging, and project root
import json
import tempfile
import time
from pathlib import Path

PROJECT = Path("/home/shumway/projects/agent-workflow-protocol")

import logging
logging.basicConfig(level=logging.INFO, format="%(name)s: %(message)s")

# Suppress noisy HTTP-level loggers
for noisy in ("httpx", "httpcore", "httpcore.http11", "httpcore.connection", "urllib3"):
    logging.getLogger(noisy).setLevel(logging.WARNING)

# Results tracker for the summary table
RESULTS: dict[str, dict] = {}

def record(name: str, passed: bool, detail: str = ""):
    """Record a test result."""
    RESULTS[name] = {"passed": passed, "detail": detail}
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {name}" + (f" -- {detail}" if detail else ""))

print("Common setup complete.")
print(f"PROJECT = {PROJECT}")
print(f"Tests will be recorded in RESULTS dict.")

---

## 2. LLM Connectivity Test

Verify that the configured LLM provider is reachable and responds correctly.
We test both plain text completion (`chat_text`) and structured JSON output (`chat_json`).

In [ ]:
from awp.runtime.llm import LLMClient

client = LLMClient(model=MODEL)
print(f"LLMClient created: model={client.model}, base_url={client.base_url}")

# --- Test chat_text ---
print("\n--- chat_text ---")
try:
    text_response = client.chat_text(
        [{"role": "user", "content": "Reply with exactly: HELLO AWP"}],
        temperature=0.0,
        max_tokens=50,
    )
    print(f"Response: {text_response!r}")
    text_ok = len(text_response.strip()) > 0
    record("2a_chat_text", text_ok, f"Got {len(text_response)} chars")
except Exception as exc:
    print(f"ERROR: {exc}")
    record("2a_chat_text", False, str(exc))

In [ ]:
# --- Test chat_json ---
print("--- chat_json ---")
try:
    json_response = client.chat_json(
        [{"role": "user", "content": (
            "Return a JSON object with exactly these keys: "
            '"name" (string, value "AWP"), "version" (string, value "1.0"), '
            '"status" (string, value "ok"). No extra text, just the JSON object.'
        )}],
        temperature=0.0,
        max_tokens=100,
    )
    print(f"Response: {json.dumps(json_response, indent=2)}")
    json_ok = (
        isinstance(json_response, dict)
        and "name" in json_response
        and "status" in json_response
    )
    record("2b_chat_json", json_ok, f"Got keys: {list(json_response.keys())}")
except Exception as exc:
    print(f"ERROR: {exc}")
    record("2b_chat_json", False, str(exc))

---

## 3. Tool Calling Test

Test the LLM's ability to use tools via `chat_with_tools`.
We register AWP's built-in arithmetic tools and ask the model to perform a calculation.
The tool-calling loop should:
1. Send the request to the LLM with tool definitions
2. LLM responds with a tool call (e.g., `arithmetic.add`)
3. AWP executes the tool and sends the result back
4. LLM produces a final text response with the answer

In [ ]:
from awp.runtime.tools import ToolRegistry

# Create a tool registry and get arithmetic tool definitions
registry = ToolRegistry()
arith_tools = registry.get_definitions(allowed=["arithmetic.*"])
print(f"Arithmetic tools available: {len(arith_tools)}")
for t in arith_tools:
    fname = t["function"]["name"]
    print(f"  - {fname}: {t['function']['description']}")

# Ask the LLM to use tools to compute 17 + 28
print("\n--- chat_with_tools: compute 17 + 28 ---")
try:
    tool_result = client.chat_with_tools(
        messages=[{"role": "user", "content": (
            "Use the arithmetic.add tool to compute 17 + 28. "
            "After getting the result, reply with the answer."
        )}],
        tools=arith_tools,
        tool_executor=lambda name, arguments: registry.call(name, arguments),
        max_rounds=5,
        temperature=0.0,
        max_tokens=200,
    )
    final_text = tool_result.get("content", "")
    print(f"Final response: {final_text!r}")

    # Verify the answer 45 appears in the response
    tools_ok = "45" in str(final_text)
    record("3_tool_calling", tools_ok, f"Expected 45 in response: {'found' if tools_ok else 'not found'}")
except Exception as exc:
    print(f"ERROR: {exc}")
    record("3_tool_calling", False, str(exc))

---

## 4. DAG Workflow -- Hello World

Run the simplest possible DAG workflow: a single agent (`greeter`) with no dependencies.
This is an A0 (Prescribed) workflow. We verify that the agent produces output with a
positive confidence score.

In [ ]:
from awp.runtime import WorkflowRunner

print("--- DAG: Hello World (01-hello-world) ---")
try:
    hello_runner = WorkflowRunner(PROJECT / "examples/01-hello-world")
    hello_result = hello_runner.run("Say hello to the AWP integration test")

    print(f"\nResult keys: {list(hello_result.keys())}")
    greeter = hello_result.get("greeter", {})
    confidence = greeter.get("confidence", 0.0)
    has_error = "error" in greeter
    print(f"Greeter confidence: {confidence}")
    if has_error:
        print(f"Greeter error: {greeter['error']}")

    # Pass if confidence > 0 (meaning the LLM responded successfully)
    hello_ok = confidence > 0 and not has_error
    record("4_dag_hello_world", hello_ok,
           f"confidence={confidence}" + (f", error={greeter.get('error', '')[:80]}" if not hello_ok else ""))

    print(f"\nFull result:")
    print(json.dumps(hello_result, indent=2, default=str))
except Exception as exc:
    print(f"ERROR: {exc}")
    record("4_dag_hello_world", False, str(exc))

---

## 5. DAG Workflow -- Research Pipeline

Run the 3-agent research pipeline (`planner -> researcher -> writer`).
This tests:
- Multi-agent DAG execution in topological order
- State sharing between agents (`share_output` fields)
- All 3 agents producing output with confidence scores

In [ ]:
print("--- DAG: Research Pipeline (02-research-pipeline) ---")
try:
    research_runner = WorkflowRunner(PROJECT / "examples/02-research-pipeline")
    research_result = research_runner.run(
        "Research the latest trends in renewable energy storage technology"
    )

    print(f"\nResult keys: {list(research_result.keys())}")

    # Check all 3 agents produced output
    expected_agents = ["planner", "researcher", "writer"]
    all_present = all(a in research_result for a in expected_agents)
    confident_count = 0

    print("\nAgent results:")
    for agent_name in expected_agents:
        agent_out = research_result.get(agent_name, {})
        conf = agent_out.get("confidence", 0.0)
        has_err = "error" in agent_out
        print(f"  {agent_name}: confidence={conf}, error={has_err}")
        if conf > 0 and not has_err:
            confident_count += 1
        # Show shared output keys (excluding confidence and error)
        data_keys = [k for k in agent_out.keys() if k not in ("confidence", "error")]
        if data_keys:
            print(f"    output keys: {data_keys}")

    # Pass if at least 2 of 3 agents produced meaningful output
    # (the writer may return non-JSON which gets wrapped with lower confidence)
    research_ok = all_present and confident_count >= 2
    record("5_dag_research_pipeline", research_ok,
           f"agents_present={all_present}, confident_agents={confident_count}/3")

except Exception as exc:
    print(f"ERROR: {exc}")
    record("5_dag_research_pipeline", False, str(exc))

In [ ]:
# Show state sharing between agents
print("=== State Sharing Analysis ===\n")
from awp.parser import parse_manifest

manifest = parse_manifest(PROJECT / "examples/02-research-pipeline/workflow.awp.yaml")

for node in manifest.orchestration.graph:
    shared_fields = node.share_output or []
    agent_output = research_result.get(node.id, {})
    deps = node.depends_on or []

    print(f"Node: {node.id}")
    print(f"  depends_on:   {deps if deps else '(root)'}")
    print(f"  share_output: {shared_fields}")
    for field in shared_fields:
        value = agent_output.get(field, "<not present>")
        text = str(value)
        if len(text) > 200:
            text = text[:200] + "..."
        print(f"  -> {field} = {text}")
    print()

---

## 6. Delegation Loop -- Simple Task

Test the delegation loop engine directly using `DelegationLoopRunner`.
We create a `DelegationLoopConfig` programmatically and run a simple
computation task. This tests:
- Manager-worker delegation pattern
- Budget enforcement (loops, workers, tokens, wall time)
- Tool calling within the delegation loop

In [ ]:
import yaml
import shutil

from awp.models.orchestration import (
    DelegationBudget,
    DelegationLoggingConfig,
    DelegationLoopConfig,
    DelegationLoopModels,
    HistoryConfig,
    StallDetectionConfig,
    ValidationConfig,
    WorkerPolicy,
    WorkerPolicyEnforced,
    SandboxEnforcement,
    CodeModeEnforcement,
    RateLimitEnforcement,
)
from awp.models.capabilities import SandboxConfig
from awp.runtime.delegation_loop_runner import DelegationLoopRunner
from awp.runtime.executor_factory import create_executor
from awp.runtime.tools import ToolRegistry

# Build config programmatically
config = DelegationLoopConfig(
    manager="agents/manager",
    models=DelegationLoopModels(
        manager=MODEL,
        worker=MODEL,
    ),
    budget=DelegationBudget(
        max_loops=5,
        max_total_workers=10,
        max_total_tokens=100_000,
        max_wall_time=300,
        max_tool_calls=50,
        max_depth=3,
    ),
    worker_policy=WorkerPolicy(
        enforced=WorkerPolicyEnforced(
            sandbox=SandboxEnforcement(type="subprocess"),
            codemode=CodeModeEnforcement(max_tools_per_worker=10),
            rate_limiting=RateLimitEnforcement(),
            forbidden_tools=["shell.execute", "file.write_outside_workspace"],
        ),
        manager_controlled=[
            "instructions", "skills", "tools_allowed",
            "output_contract", "codemode.enabled", "codemode.tool_creation",
        ],
    ),
    termination=StallDetectionConfig(
        enabled=True,
        window=3,
        min_confidence_delta=0.05,
        action="warn_then_stop",
    ),
    validation=ValidationConfig(),
    history=HistoryConfig(
        rolling_summary=True,
        full_results_window=3,
        persist_to_disk=True,
    ),
    logging=DelegationLoggingConfig(
        format="dual",
        persist_artifacts=True,
    ),
)

print(f"Config created:")
print(f"  Budget: loops={config.budget.max_loops}, workers={config.budget.max_total_workers}, "
      f"tokens={config.budget.max_total_tokens:,}, wall_time={config.budget.max_wall_time}s")
print(f"  Models: manager={config.models.manager}, worker={config.models.worker}")

In [ ]:
# Set up workspace directory with a manager agent
workspace_dir = Path(tempfile.mkdtemp(prefix="awp_e2e_deleg_simple_"))
(workspace_dir / "workspace").mkdir(exist_ok=True)

# Write manager agent config
manager_dir = workspace_dir / "agents" / "manager"
manager_dir.mkdir(parents=True, exist_ok=True)

system_prompt = """You are a task delegation manager.
You receive a task and delegate it to workers.
Each worker can execute Python code or use arithmetic tools.
Collect worker results and produce a final JSON answer.
"""
(manager_dir / "system_prompt.md").write_text(system_prompt)

agent_config = {
    "awp_agent": "1.0.0",
    "identity": {
        "id": "manager",
        "role": "Task Manager",
        "version": "1.0.0",
        "description": "Manager agent for delegation loop integration test",
    },
    "runtime": {"class_name": "Agent", "strategy_folder": "workflow"},
    "model": {"name": MODEL, "temperature": 0.2, "max_tokens": 4096},
    "prompt": {"system": "system_prompt.md"},
    "output": {"format": "json"},
}
(manager_dir / "agent.awp.yaml").write_text(
    yaml.dump(agent_config, default_flow_style=False, allow_unicode=True)
)

# Create tool registry and code executor
tool_registry = ToolRegistry(workflow_dir=workspace_dir)
sandbox_cfg = SandboxConfig(enabled=True, type="subprocess")
code_executor = create_executor(sandbox_cfg, working_dir=workspace_dir / "workspace")
tool_registry.set_code_executor(code_executor)

print(f"Workspace: {workspace_dir}")
print(f"Tool registry: {len(tool_registry.get_definitions())} tools available")

In [ ]:
# Run the delegation loop
print("--- Delegation Loop: Simple Task ---")
try:
    runner = DelegationLoopRunner(
        workflow_dir=workspace_dir,
        config=config,
        tool_registry=tool_registry,
        manager_model=MODEL,
        worker_model=MODEL,
    )

    task = "Calculate the sum of integers from 1 to 100 and return the result as JSON."
    print(f"Task: {task}\n")

    deleg_result = runner.run(task)

    print(f"\n=== Delegation Loop Result ===")
    print(json.dumps(deleg_result, indent=2, default=str)[:2000])

    # Show budget consumption
    budget = runner._budget
    print(f"\n=== Budget Consumption ===")
    print(f"  Loops used:      {budget.loops_used}")
    print(f"  Workers spawned: {budget.workers_spawned}")
    print(f"  Tokens consumed: {budget.tokens_consumed:,}")
    print(f"  Wall time:       {budget.wall_time_elapsed:.1f}s")
    print(f"  Tool calls used: {budget.tool_calls_used}")

    # Pass if we got some result and did not error out completely
    deleg_ok = (
        isinstance(deleg_result, dict)
        and budget.loops_used > 0
    )
    record("6_delegation_simple", deleg_ok,
           f"loops={budget.loops_used}, workers={budget.workers_spawned}, "
           f"tool_calls={budget.tool_calls_used}")

except Exception as exc:
    print(f"ERROR: {exc}")
    import traceback; traceback.print_exc()
    record("6_delegation_simple", False, str(exc)[:200])

---

## 7. Delegation Loop -- Tool Mode (code.execute)

Test the delegation loop with code execution enabled.
The config includes `codemode.enabled` and `codemode.tool_creation` in `manager_controlled`,
which allows the manager to instruct workers to execute Python code.
We verify that `code.execute` is actually used and `tool_calls_used` is tracked.

In [ ]:
# Set up a fresh workspace for code execution test
workspace_code = Path(tempfile.mkdtemp(prefix="awp_e2e_deleg_code_"))
(workspace_code / "workspace").mkdir(exist_ok=True)

# Write manager agent config for code-execution-capable delegation
manager_code_dir = workspace_code / "agents" / "manager"
manager_code_dir.mkdir(parents=True, exist_ok=True)

code_system_prompt = """You are a Python code execution manager.
You delegate tasks to workers who can run Python code using the code.execute tool.
Always instruct workers to use code.execute to solve problems programmatically.
Collect results and produce a final JSON answer.
"""
(manager_code_dir / "system_prompt.md").write_text(code_system_prompt)

code_agent_config = {
    "awp_agent": "1.0.0",
    "identity": {
        "id": "manager",
        "role": "Code Execution Manager",
        "version": "1.0.0",
        "description": "Manager that delegates code execution tasks",
    },
    "runtime": {"class_name": "Agent", "strategy_folder": "workflow"},
    "model": {"name": MODEL, "temperature": 0.2, "max_tokens": 4096},
    "prompt": {"system": "system_prompt.md"},
    "output": {"format": "json"},
}
(manager_code_dir / "agent.awp.yaml").write_text(
    yaml.dump(code_agent_config, default_flow_style=False, allow_unicode=True)
)

# Build config with code mode enabled
code_config = DelegationLoopConfig(
    manager="agents/manager",
    models=DelegationLoopModels(manager=MODEL, worker=MODEL),
    budget=DelegationBudget(
        max_loops=5,
        max_total_workers=10,
        max_total_tokens=100_000,
        max_wall_time=300,
        max_tool_calls=50,
        max_depth=3,
    ),
    worker_policy=WorkerPolicy(
        enforced=WorkerPolicyEnforced(
            sandbox=SandboxEnforcement(type="subprocess"),
            codemode=CodeModeEnforcement(max_tools_per_worker=10),
            rate_limiting=RateLimitEnforcement(),
            forbidden_tools=["shell.execute", "file.write_outside_workspace"],
        ),
        manager_controlled=[
            "instructions", "skills", "tools_allowed",
            "output_contract", "codemode.enabled", "codemode.tool_creation",
        ],
    ),
    termination=StallDetectionConfig(enabled=True, window=3, min_confidence_delta=0.05, action="warn_then_stop"),
    validation=ValidationConfig(),
    history=HistoryConfig(rolling_summary=True, full_results_window=3, persist_to_disk=True),
    logging=DelegationLoggingConfig(format="dual", persist_artifacts=True),
)

# Tool registry with code executor
code_tool_registry = ToolRegistry(workflow_dir=workspace_code)
code_sandbox_cfg = SandboxConfig(enabled=True, type="subprocess")
code_exec = create_executor(code_sandbox_cfg, working_dir=workspace_code / "workspace")
code_tool_registry.set_code_executor(code_exec)

print(f"Workspace: {workspace_code}")
print(f"Code executor type: {type(code_exec).__name__}")
print(f"Tools available: {len(code_tool_registry.get_definitions())}")

In [ ]:
# Run the code-execution delegation loop
print("--- Delegation Loop: Code Execution Task ---")
try:
    code_runner = DelegationLoopRunner(
        workflow_dir=workspace_code,
        config=code_config,
        tool_registry=code_tool_registry,
        manager_model=MODEL,
        worker_model=MODEL,
    )

    code_task = (
        "Write and execute Python code to compute the first 10 Fibonacci numbers. "
        "Use the code.execute tool. Return the list as JSON."
    )
    print(f"Task: {code_task}\n")

    code_result = code_runner.run(code_task)

    print(f"\n=== Code Execution Result ===")
    print(json.dumps(code_result, indent=2, default=str)[:2000])

    # Show budget consumption
    code_budget = code_runner._budget
    print(f"\n=== Budget Consumption ===")
    print(f"  Loops used:      {code_budget.loops_used}")
    print(f"  Workers spawned: {code_budget.workers_spawned}")
    print(f"  Tokens consumed: {code_budget.tokens_consumed:,}")
    print(f"  Wall time:       {code_budget.wall_time_elapsed:.1f}s")
    print(f"  Tool calls used: {code_budget.tool_calls_used}")

    # Pass if tool_calls were used (indicating code.execute was invoked)
    code_ok = (
        isinstance(code_result, dict)
        and code_budget.loops_used > 0
        and code_budget.tool_calls_used > 0
    )
    record("7_delegation_code_execute", code_ok,
           f"loops={code_budget.loops_used}, tool_calls={code_budget.tool_calls_used}")

except Exception as exc:
    print(f"ERROR: {exc}")
    import traceback; traceback.print_exc()
    record("7_delegation_code_execute", False, str(exc)[:200])

---

## Test 8: AgentWorkflow — DataFrame + Source Inputs

Test `AgentWorkflow` with mixed inputs: inline DataFrame + `Source.sql()` (SQLite query)
+ `Source.base64()` (decoded config). All Source objects are resolved in parallel before
the workflow starts. Agents see native Python objects, not raw URLs or queries.

In [ ]:
import pandas as pd
import base64
import sqlite3
from awp.data import AgentWorkflow, Source

# Create sample DataFrame (inline input)
df = pd.DataFrame({
    "date": pd.date_range("2024-01-01", periods=30, freq="D"),
    "temperature": [20 + i * 0.5 + (i % 7) * 2 for i in range(30)],
    "humidity": [60 - i * 0.3 + (i % 5) * 3 for i in range(30)],
    "city": ["Berlin", "Munich", "Hamburg"] * 10,
})

# Create a temporary SQLite DB with supplementary data (Source.sql input)
db_path = Path(tempfile.mkdtemp()) / "weather_meta.db"
conn = sqlite3.connect(str(db_path))
conn.execute("CREATE TABLE city_info (city TEXT, country TEXT, population INT)")
conn.executemany(
    "INSERT INTO city_info VALUES (?, ?, ?)",
    [("Berlin", "Germany", 3645000), ("Munich", "Germany", 1472000),
     ("Hamburg", "Germany", 1841000)],
)
conn.commit()
conn.close()

# Encode analysis hints as base64 (Source.base64 input)
hints_b64 = base64.b64encode(
    b'{"focus_metric": "temperature", "compare_by": "city"}'
).decode()

print(f"Input DataFrame: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")
print(f"Cities: {df['city'].unique().tolist()}")
print(f"SQLite DB: {db_path}")
print(f"Base64 hints: {len(hints_b64)} chars")
print()

# Set up output directory
df_output_dir = Path(tempfile.mkdtemp(prefix="awp_e2e_df_analysis_"))

print("--- AgentWorkflow: DataFrame + Source.sql + Source.base64 ---")
try:
    df_result = AgentWorkflow(
        inputs={
            "weather_data": df,
            "city_metadata": Source.sql(
                "SELECT * FROM city_info",
                dsn=f"sqlite:///{db_path}",
            ),
            "analysis_hints": Source.base64(hints_b64, format="text"),
        },
        task=(
            "Analyze the weather data: calculate the average temperature per city, "
            "find the hottest and coldest days, and summarize the trends. "
            "Use city_metadata for population context and analysis_hints for focus."
        ),
        model=MODEL,
        max_loops=5,
        max_wall_time=300,
        max_total_tokens=200_000,
        code_mode=True,
        tool_creation=False,
        output_dir=str(df_output_dir),
        verbose=True,
    ).run()

    print(f"\n=== AgentWorkflow Result ===")
    print(f"Status: {df_result.get('status', 'unknown')}")

    metadata = df_result.get("metadata", {})
    print(f"\n=== Metadata ===")
    for k, v in metadata.items():
        print(f"  {k}: {v}")

    print(f"\n=== Result (first 1000 chars) ===")
    result_str = json.dumps(df_result.get("result", {}), indent=2, default=str)
    print(result_str[:1000])

    # Pass if status indicates completion (complete/partial with results)
    status = df_result.get("status", "")
    df_ok = status in ("complete", "completed", "budget_exceeded", "partial") and "metadata" in df_result
    loops = metadata.get("loops", 0)
    wall = metadata.get("wall_time", 0)
    tools = metadata.get("tool_calls", 0)
    record("8_agentworkflow_dataframe_source", df_ok,
           f"status={status}, loops={loops}, wall_time={wall:.1f}s, tool_calls={tools}")

except Exception as exc:
    print(f"ERROR: {exc}")
    import traceback; traceback.print_exc()
    record("8_agentworkflow_dataframe_source", False, str(exc)[:200])

---

## 9. AgentWorkflow -- Tool Creation

Test the highest autonomy feature: dynamic tool creation at runtime.
With `tool_creation=True`, workers can create new tools that persist
for subsequent iterations. This is what enables A3-A4 autonomy levels.

In [ ]:
tool_output_dir = Path(tempfile.mkdtemp(prefix="awp_e2e_tool_creation_"))

print("--- AgentWorkflow: Tool Creation ---")
try:
    tool_result = AgentWorkflow(
        inputs={"values": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]},
        task=(
            "Create a custom tool that calculates fibonacci numbers, "
            "then use it to compute fib(10). Return the result as JSON."
        ),
        model=MODEL,
        code_mode=True,
        tool_creation=True,
        max_loops=5,
        max_wall_time=600,
        max_total_tokens=200_000,
        output_dir=str(tool_output_dir),
        verbose=True,
    ).run()

    print(f"\n=== Tool Creation Result ===")
    print(f"Status: {tool_result.get('status', 'unknown')}")

    metadata = tool_result.get("metadata", {})
    print(f"\n=== Metadata ===")
    for k, v in metadata.items():
        print(f"  {k}: {v}")

    print(f"\n=== Result (first 1000 chars) ===")
    result_str = json.dumps(tool_result.get("result", {}), indent=2, default=str)
    print(result_str[:1000])

    # Check for tools_created in metadata or result
    tools_created = metadata.get("tools_created", 0)
    print(f"\nTools created: {tools_created}")

    # Pass if workflow ran (any status except error with 0 loops)
    status = tool_result.get("status", "")
    loops = metadata.get("loops", 0)
    tool_ok = status in ("complete", "completed", "budget_exceeded", "partial", "stall_detected") and loops > 0
    record("9_agentworkflow_tool_creation", tool_ok,
           f"status={status}, tools_created={tools_created}, loops={loops}")

except Exception as exc:
    print(f"ERROR: {exc}")
    import traceback; traceback.print_exc()
    record("9_agentworkflow_tool_creation", False, str(exc)[:200])

---

## 10. Summary -- Pass/Fail Table

Collect all test results and display a summary table showing which capabilities
passed and which failed.

In [ ]:
# ============================================================
# End-to-End Integration Test Summary
# ============================================================

print("=" * 80)
print("         AWP End-to-End Integration Test Summary")
print("=" * 80)
print()

# Friendly names for each test
test_names = {
    "2a_chat_text":                  "LLM Connectivity: chat_text",
    "2b_chat_json":                  "LLM Connectivity: chat_json (structured)",
    "3_tool_calling":                "Tool Calling: arithmetic via chat_with_tools",
    "4_dag_hello_world":             "DAG Workflow: Hello World (A0)",
    "5_dag_research_pipeline":       "DAG Workflow: Research Pipeline (A1)",
    "6_delegation_simple":           "Delegation Loop: Simple Task (A2)",
    "7_delegation_code_execute":     "Delegation Loop: Code Execution (A3)",
    "8_agentworkflow_dataframe":     "AgentWorkflow: DataFrame Analysis",
    "9_agentworkflow_tool_creation": "AgentWorkflow: Tool Creation (A4)",
}

passed = 0
failed = 0

print(f"{'#':<4s} {'Test':<50s} {'Status':<8s} {'Detail'}")
print("-" * 80)

for i, (key, friendly) in enumerate(test_names.items(), 1):
    result = RESULTS.get(key, {"passed": False, "detail": "NOT RUN"})
    status = "PASS" if result["passed"] else "FAIL"
    detail = result.get("detail", "")
    if len(detail) > 50:
        detail = detail[:47] + "..."
    print(f"{i:<4d} {friendly:<50s} {status:<8s} {detail}")
    if result["passed"]:
        passed += 1
    else:
        failed += 1

total = passed + failed
print("-" * 80)
print(f"\nTotal: {total} tests | Passed: {passed} | Failed: {failed}")
print(f"Pass rate: {passed/total*100:.0f}%" if total > 0 else "No tests run")
print()

if failed == 0:
    print("ALL TESTS PASSED -- AWP end-to-end integration is fully functional.")
else:
    print(f"WARNING: {failed} test(s) failed. Check the details above for diagnostics.")